# Notebook 3: Adding Instrumental Realism to S-PLUS Mocks
**Author:** Gustavo Fernandes Gonçalves
ORCID: [0009-0006-5887-6621](https://orcid.org/0009-0006-5887-6621)
Lattes: http://lattes.cnpq.br/7416443230735445

Start with the *noise-free* FITS files from `fits_noisefree/{subhalo_id}/`. 

A: pixel resampling. 

B: unit (brightness → electrons). 

C: PSF. 

D: noise. 

**One filter at a time**: change `filter_name` at the top and repeat the cells.

## 1. Selecting a filter and opening files

In [ ]:
from astropy.io import fits
from astropy.visualization import PercentileInterval, AsinhStretch, ImageNormalize
import numpy as np
import matplotlib.pyplot as plt
 
from tutorial_tools import (
    get_splus_filters, load_transmission, connect_ds9,
    compute_target_pixels, photon_gain_factor, surface_brightness_to_counts,
    apply_psf, apply_noise,
)
 
subhalo_id = '10'  # <- replace with your subhalo ID
filter_name = 'g'  # <- replace with any of the 12 filters
 
hdu = fits.open(f'fits_noisefree/{subhalo_id}/{subhalo_id}_{filter_name}.fits')
img = hdu[0].data
header = hdu[0].header
 
plt.imshow(img, origin='lower', cmap='gray', norm=ImageNormalize(img, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
plt.title(f'{filter_name} (noise-free)')

## 2. Part A — Resampling

The simulation's physical field of view is fixed (60 kpc), but the number of pixels it occupies in the sky depends on the angular diameter distance, which is a function of redshift. We calculate the angle subtended by this field of view and divide it by the actual pixel scale of [S-PLUS (0.55 arcsec/pixel)](https://www.splus.iag.usp.br/instrumentation/) to determine how many pixels the final image should have.

$$\theta = \frac{\text{physical FOV}}{D_A(z)}, \qquad N_{\text{pixels}} = \frac{\theta}{\text{pixel scale}}$$

In [ ]:
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
 
cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089, Ob0=0.0486)
 
z = 0.01  # <- object redshift
field_phys = 60 * u.kpc  # physical FOV
splus_pixel_scale = 0.55 * u.arcsec  # S-PLUS value

In [ ]:
d_ang, theta_arcsec, num_pixels = compute_target_pixels(z, field_phys, splus_pixel_scale, cosmo)
 
print(f'Angular distance: {d_ang:.1f}')
print(f'FOV: {theta_arcsec:.2f}')
print(f'Number of pixels: {num_pixels} x {num_pixels}')

In [ ]:
from scipy.ndimage import zoom
 
zoom_factor = num_pixels / img.shape[0]
img_rebinned = zoom(img, zoom_factor, order=1)
img_rebinned[img_rebinned < 0] = 0
 
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 4))
axL.imshow(img, origin='lower', cmap='gray', norm=ImageNormalize(img, interval=PercentileInterval(99.5),stretch=AsinhStretch(0.03)))
axL.set_title(f'Original ({img.shape[0]}x{img.shape[1]} px)')
axR.imshow(img_rebinned, origin='lower', cmap='gray', norm=ImageNormalize(img_rebinned, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
axR.set_title(f'Resampled ({img_rebinned.shape[0]}x{img_rebinned.shape[1]} px)')
plt.tight_layout()

## 3. Part B — Unit

| Parameter | Value | Source |
|---|---|---|
| Aperture | 0.826 m | [instrumentation](https://www.splus.iag.usp.br/instrumentation/) |
| N exposures | 3 | [Mendes de Oliveira et al. 2019, Tab. 5](https://arxiv.org/abs/1907.01567) |
| t_exp per filter | see definition below | same table |
| Sky background per band | see definition below | Gustavo Schwarz, S-PLUS data reduction team (private communication) |


Sky background estimate: median, across many science frames, of the modal sky level measured in 128x128 pixel boxes across each image ("median of medians")

In [ ]:
exposure_times = {
    'u': 227, 'J0378': 220, 'J0395': 118, 'J0410': 59, 'J0430': 57, 'g': 33,
    'J0515': 61, 'r': 40, 'J0660': 290, 'i': 46, 'J0861': 80, 'z': 56,
}

sky_bkg_per_band = {
    'u': 0.1055, 'J0378': 0.0499, 'J0395': 0.0443, 'J0410': 0.1437, 'J0430': 0.1667, 'g': 1.7977,
    'J0515': 0.2722, 'r': 3.9129, 'J0660': 0.2996, 'i': 6.3343, 'J0861': 2.4548, 'z': 9.8222,
}  # [e-/s]

n_exp = 3
aperture_m = 0.826

In [ ]:
base = './filter_curves-master'
 
filters = get_splus_filters(base)

Converting surface brightness to detected photons. Physical gain factor, Eq. C.4 from [Zhou et al. 2025, A&A, 700, A120](https://arxiv.org/abs/2506.15060),  which cites [Ryon 2023, *ACS Instrument Handbook for Cycle 31*, Seção 9.2](https://hst-docs.stsci.edu/acsihb) (HST/ACS instrument handbook):
 
$$g_{\text{phys}} = \frac{h}{A_{\text{aper}} \displaystyle\int \frac{R_\lambda}{\lambda}\, d\lambda}$$

In [ ]:
g_phys = photon_gain_factor(aperture_m, filters[filter_name])
t_exp = exposure_times[filter_name]
 
counts_signal = surface_brightness_to_counts(img_rebinned, splus_pixel_scale, n_exp, t_exp, g_phys)
 
plt.imshow(counts_signal, origin='lower', cmap='gray', norm=ImageNormalize(counts_signal, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
plt.colorbar(label='e-')
plt.title(f'{filter_name} (signal in electrons)')

## 4. Part C — PSF

$$\sigma = \frac{\text{FWHM}}{2\sqrt{2\ln 2}}$$
 
Typical S-PLUS FWHM: ~1.5 arcsec, range 0.8–2.0  ([Mendes de Oliveira et al. 2019, Tab. 9](https://arxiv.org/abs/1907.01567)).

In [ ]:
fwhm_arcsec = 0.8  # <- 0.8-2.0 arcsec
 
counts_signal_psf = apply_psf(counts_signal, fwhm_arcsec, splus_pixel_scale)
 
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 4))
axL.imshow(counts_signal, origin='lower', cmap='gray', norm=ImageNormalize(counts_signal, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
axL.set_title('Before PSF')
axR.imshow(counts_signal_psf, origin='lower', cmap='gray', norm=ImageNormalize(counts_signal_psf, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
axR.set_title('After PSF')
plt.tight_layout()

## 5. Part D — Noise
Read noise: 3.43 e⁻/exposure ([Mendes de Oliveira et al. 2019, Tab. 1](https://arxiv.org/abs/1907.01567)). 

Sky background per band (Gustavo Schwarz, S-PLUS data reduction team (private communication))

In [ ]:
read_noise = 3.43  # e-, per exposure
sky_bkg = sky_bkg_per_band[filter_name]  # [e-/s]
dark_current = 0.0   # [e-/s] — negligible
 
counts_final = apply_noise(counts_signal_psf, n_exp, t_exp, sky_bkg, dark_current, read_noise)
 
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 4))
axL.imshow(img, origin='lower', cmap='gray', norm=ImageNormalize(img, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
axL.set_title('Original (noise-free)')
axR.imshow(counts_final, origin='lower', cmap='gray', norm=ImageNormalize(counts_final, interval=PercentileInterval(99.5), stretch=AsinhStretch(0.03)))
axR.set_title('Ressampled + PSF + noise')
plt.tight_layout()

## 6. Saving the result

In [ ]:
import os
 
outdir = f'fits_mock_images/{subhalo_id}'
os.makedirs(outdir, exist_ok=True)
 
hdr = fits.Header()
hdr['FILTER'] = filter_name
hdr['REDSHIFT'] = z
hdr['PIXSCAL'] = splus_pixel_scale.to(u.arcsec).value
hdr['TEXP'] = t_exp
hdr['NEXP'] = n_exp
hdr['FWHMPSF'] = fwhm_arcsec
hdr['BUNIT'] = 'electron'
 
fits.PrimaryHDU(counts_final.astype('float32'), header=hdr).writeto(f'{outdir}/{subhalo_id}_{filter_name}.fits', overwrite=True)

## 7. Comparison on DS9

`sudo apt install saods9 xpa-tools` and `pip install pyds9` required. The first time you use it, you may need to register it under **File → XPA → Connect**.

In [ ]:
d = connect_ds9()

In [ ]:
noisefree_path = f'fits_noisefree/{subhalo_id}/{subhalo_id}_{filter_name}.fits'
mock_path = f'{outdir}/{subhalo_id}_{filter_name}.fits'
 
d.set('frame delete all')
d.set('tile yes')
 
d.set('frame new')
d.set(f'file {noisefree_path}')
 
d.set('frame new')
d.set(f'file {mock_path}')

Close all existing frames and open only the two FITS files: the original noise-free one and the final mock generated in this section.

## 8. Repeat for the other filters

Go back to cell 1, change `filter_name` to another one of the 12 filters, and run all the cells again in sequence